In [189]:
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from FlagEmbedding import BGEM3FlagModel
import torch
import numpy as np

### Config

In [201]:
from pathlib import Path

# --- 1. Path & Connection Configuration ---
BASE_DIR = Path.cwd().parent
DATASET_PATH = BASE_DIR / "data" / "processed" / "nitibench_cleaned_2026-03-17.parquet"
EMBEDDING_PATH = BASE_DIR / "data" / "processed" / "vectors_sparse_nitibench-ccl-bge-m3.parquet" # embedding

# เปลี่ยนจาก path เป็น url ของ localhost
QDRANT_URL = "http://localhost:6333" 
COLLECTION_NAME = "thai_laws_collection"
MODEL_NAME = "VISAI-AI/nitibench-ccl-human-finetuned-bge-m3"

### Initial Qdrant Cient and Embedding Model

In [202]:
# --- 2. Initialize Client & Model ---
client = QdrantClient(url=QDRANT_URL, timeout=60) # เชื่อมต่อผ่าน URL

# เช็คว่ามี GPU (CUDA) หรือไม่
# ถ้ามีให้ใช้ 'cuda' ถ้าไม่มีให้ใช้ 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = True if device == 'cuda' else False  # fp16 ส่วนใหญ่รองรับเฉพาะบน GPU

print(f"กำลังรันโมเดลบน: {device.upper()}")

# 2. Initialize Model
model = BGEM3FlagModel(
    'VISAI-AI/nitibench-ccl-human-finetuned-bge-m3', 
    use_fp16=use_fp16, 
    device=device 
)

กำลังรันโมเดลบน: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<00:00, 4209.13it/s]


In [203]:
# วิธีตรวจสอบ
print(f"Device being used: {model.device}")

# อีกวิธีคือเช็คผ่าน torch โดยตรง (ถ้า model ใช้ torch เป็น backend)
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Current GPU device name: {torch.cuda.get_device_name(0)}")

Device being used: cuda
Is CUDA available? True
Current GPU device name: NVIDIA GeForce GTX 1650


### Create Collection in Qdrant

In [204]:
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, Modifier

# --- 3. Create Collection (Updated for Hybrid Search) ---
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        # เปลี่ยนจาก VectorParams เป็น dict เพื่อระบุชื่อ 'dense'
        vectors_config={
            "dense": VectorParams(size=1024, distance=Distance.COSINE)
        },
        # เปลี่ยน key จาก "lexical" เป็น "sparse"
        sparse_vectors_config={
            "sparse": SparseVectorParams(
                modifier=Modifier.IDF
            ) 
        }
    )
    print(f"สร้าง Collection: {COLLECTION_NAME} รองรับ Hybrid Search เรียบร้อย")

สร้าง Collection: thai_laws_collection รองรับ Hybrid Search เรียบร้อย


### Embedding Dense Vector and Sparse

### save as parquet

look at ```embedding.py```

In [225]:
import pandas as pd
df_backup = pd.read_parquet(EMBEDDING_PATH)
df_backup.head()

,id,law_name,section_num,text,dense_vector,sparse_vector
0,0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,132,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,"[-0.03570556640625, 0.004070281982421001, -0.0...","{""221767"": 0.060760498046875, ""70390"": 0.15673..."
1,1,ประมวลกฎหมายแพ่งและพาณิชย์,1598/5,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,"[0.0345458984375, 0.049072265625, -0.020095825...","{""130047"": 0.0031299591064453125, ""57467"": 0.0..."
2,2,ประมวลกฎหมายแพ่งและพาณิชย์,876,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,"[-0.039886474609375, 0.05941772460937501, 0.00...","{""167120"": 0.015411376953125, ""382"": 0.0497741..."
3,3,ประมวลกฎหมายแพ่งและพาณิชย์,1030,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,"[-0.029144287109375003, 0.024978637695312, -0....","{""130047"": 0.11090087890625, ""57467"": 0.025955..."
4,4,ประมวลกฎหมายแพ่งและพาณิชย์,849,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,"[0.011123657226562, 0.0675048828125, -0.041503...","{""130047"": 0.00732421875, ""57467"": 0.020690917..."


### Load Embedding Parquet and Ingest to Qdrant

In [220]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from qdrant_client.models import PointStruct, SparseVector

# === Merge metadata ===
original_dataset_path: Path = BASE_DIR / "data" / "raw" / 'nitibench' / 'data' / 'nitibench-ccl.parquet'

df_backup: pd.DataFrame = pd.read_parquet(EMBEDDING_PATH)
# columns: id(int), text(str), law_name(str), section_num(str),
#          dense_vector(np.ndarray[float32]), sparse_vector(str|dict)

original_df: pd.DataFrame = pd.read_parquet(original_dataset_path)
# columns: relevant_laws(np.ndarray[dict]), reference_laws(np.ndarray[dict])

def extract_law_info(row: np.ndarray, key: str) -> str | None:
    if row.size > 0:
        return row[0].get(key)
    return None

original_df["law_name"] = original_df["relevant_laws"].apply(extract_law_info, key="law_name")
# dtype: str | None
original_df["section_num"] = original_df["relevant_laws"].apply(extract_law_info, key="section_num")
# dtype: str | None

original_slim: pd.DataFrame = (
    original_df[["law_name", "section_num", "reference_laws"]]
    # reference_laws dtype: np.ndarray[dict{"law_name": str, "section_num": str, "section_content": str}]
    .drop_duplicates(subset=["law_name", "section_num"])
)

merge_df: pd.DataFrame = pd.merge(df_backup, original_slim, on=["law_name", "section_num"], how='left')
# reference_laws dtype after merge: np.ndarray[dict] | float(NaN)

# Replace NaN → np.array([])
is_nan_mask: pd.Series = merge_df["reference_laws"].isna()  # dtype: bool
empty_series: pd.Series = pd.Series(
    [np.array([]) for _ in range(is_nan_mask.sum())],
    index=merge_df[is_nan_mask].index,
    dtype=object
)
merge_df.loc[is_nan_mask, 'reference_laws'] = empty_series
# reference_laws dtype: np.ndarray[dict] | np.ndarray([])

# convert ndarray → list[dict] (ทั้ง non-empty และ empty array)
merge_df["reference_laws"] = merge_df["reference_laws"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)
# reference_laws dtype: list[dict{"law_name": str, "section_num": str, "section_content": str}] | list[]

def slim_ref_laws(row: list[dict]) -> list[dict]:
    if len(row) > 0:
        return [{"law_name": law["law_name"], "section_num": law["section_num"]} for law in row]
    return row

merge_df["reference_laws"] = merge_df["reference_laws"].apply(slim_ref_laws)
# reference_laws dtype: list[dict{"law_name": str, "section_num": str}] | list[]

# Sanity check ก่อน ingest
assert len(merge_df) == len(df_backup), f"Row count mismatch! merge_df={len(merge_df)}, df_backup={len(df_backup)}"

# === Ingestion ===
all_points: list[PointStruct] = []

for _, row in merge_df.iterrows():
    sparse_raw: str | dict = row['sparse_vector']
    if isinstance(sparse_raw, str):
        sparse_data: dict[str, float | None] = json.loads(sparse_raw)
    else:
        sparse_data: dict[str, float | None] = sparse_raw

    indices: list[int] = []
    values: list[float] = []
    for k, v in sparse_data.items():
        if v is not None:
            indices.append(int(k))
            values.append(float(v))

    vectors_dict: dict[str, list[float] | SparseVector] = {
        "dense": row['dense_vector'].tolist() if hasattr(row['dense_vector'], 'tolist') else row['dense_vector']
        # dense dtype: list[float32]
    }

    if indices:
        vectors_dict["sparse"] = SparseVector(indices=indices, values=values)

    all_points.append(
        PointStruct(
            id=int(row['id']),
            vector=vectors_dict,
            payload={
                "text": row['text'],                      # str
                "law_name": row['law_name'],              # str
                "section_num": row['section_num'],        # str
                "reference_laws": row['reference_laws']   # list[dict{"law_name": str, "section_num": str}] | list[]
            }
        )
    )

print(f"Total points created: {len(all_points)}")

Total points created: 3833


In [ ]:
# DATASET_REF_PATH = BASE_DIR / "data" / "processed" / "ref_vectors_sparse_nitibench-ccl-bge-m3.parquet"
# merge_df.to_parquet(DATASET_REF_PATH, engine='pyarrow', index=False)

In [222]:
# --- ขั้นตอนที่ 2: เริ่มส่งข้อมูลเข้า Qdrant (Ingestion) ---
# มั่นใจได้ว่าข้อมูลทุกอย่างพร้อมแล้วใน all_points
# ตัวอย่างการทำ Batching แบบง่าย
batch_size = 100
for i in range(0, len(all_points), batch_size):
    batch = all_points[i : i + batch_size]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch
    )
    print(f"Uploaded {i + len(batch)} points...")

Uploaded 100 points...
Uploaded 200 points...
Uploaded 300 points...
Uploaded 400 points...
Uploaded 500 points...
Uploaded 600 points...
Uploaded 700 points...
Uploaded 800 points...
Uploaded 900 points...
Uploaded 1000 points...
Uploaded 1100 points...
Uploaded 1200 points...
Uploaded 1300 points...
Uploaded 1400 points...
Uploaded 1500 points...
Uploaded 1600 points...
Uploaded 1700 points...
Uploaded 1800 points...
Uploaded 1900 points...
Uploaded 2000 points...
Uploaded 2100 points...
Uploaded 2200 points...
Uploaded 2300 points...
Uploaded 2400 points...
Uploaded 2500 points...
Uploaded 2600 points...
Uploaded 2700 points...
Uploaded 2800 points...
Uploaded 2900 points...
Uploaded 3000 points...
Uploaded 3100 points...
Uploaded 3200 points...
Uploaded 3300 points...
Uploaded 3400 points...
Uploaded 3500 points...
Uploaded 3600 points...
Uploaded 3700 points...
Uploaded 3800 points...
Uploaded 3833 points...
